# Light Rail Walkshed Explorer

Interactive map of Sound Transit 1 Line + 2 Line stations with 10-minute walking isochrones from Mapbox.

In [1]:
import csv
import json
import time
from pathlib import Path

import geopandas as gpd
import networkx as nx
from osmnx import convert, distance, features, graph
from shapely.geometry import mapping, MultiPoint
from shapely.ops import unary_union

PROJECT_ROOT = Path.cwd().parent
GTFS_DIR = PROJECT_ROOT / "data" / "gtfs"
OUTPUT = PROJECT_ROOT / "data" / "stations.geojson"

TARGET_ROUTES = {"100479", "2LINE"}  # 1 Line, 2 Line
WALK_SPEED_M_PER_MIN = 75  # ~4.5 km/h
CONTOUR_MINUTES = [5, 10, 15]

def load_stations() -> list[dict]:
    """Deduplicated 1 Line + 2 Line stations from GTFS."""
    trips_path = GTFS_DIR / "40" / "trips.txt"
    stop_times_path = GTFS_DIR / "40" / "stop_times.txt"
    stops_path = GTFS_DIR / "40" / "stops.txt"
    for p in [trips_path, stop_times_path, stops_path]:
        if not p.exists():
            raise FileNotFoundError(f"{p} not found. Run scripts/download_gtfs.py first.")

    with open(trips_path) as f:
        trip_ids = {t["trip_id"] for t in csv.DictReader(f) if t["route_id"] in TARGET_ROUTES}
    with open(stop_times_path) as f:
        stop_ids = {r["stop_id"] for r in csv.DictReader(f) if r["trip_id"] in trip_ids}

    seen: set[str] = set()
    stations = []
    with open(stops_path) as f:
        for s in csv.DictReader(f):
            if s["stop_id"] in stop_ids and s["stop_name"] not in seen:
                seen.add(s["stop_name"])
                stations.append({
                    "name": s["stop_name"],
                    "lat": float(s["stop_lat"]),
                    "lon": float(s["stop_lon"]),
                })
    return sorted(stations, key=lambda s: s["name"])

print(f"Project root: {PROJECT_ROOT}")
stations = load_stations()
print(f"Found {len(stations)} light rail stations")

Project root: /Users/tommydoerr/dev/transit_tracker-data-refinement
Found 38 light rail stations


## Generate Walksheds via OSMnx

Downloads the OpenStreetMap walking network around each station, then computes network-based isochrones at 5/10/15 minutes. Results are cached to `data/stations.geojson`.

In [ ]:
def compute_walksheds(lat: float, lon: float, contour_minutes: list[int] = CONTOUR_MINUTES) -> list[dict]:
    """Compute walking isochrone polygons using OSM network data.
    
    Returns list of {"minutes": int, "geometry": GeoJSON geometry dict}.
    """
    max_dist = max(contour_minutes) * WALK_SPEED_M_PER_MIN * 1.4  # buffer for network detours
    G = graph.graph_from_point((lat, lon), dist=max_dist, network_type="walk")
    distance.add_edge_lengths(G)
    center = distance.nearest_nodes(G, lon, lat)

    results = []
    for minutes in sorted(contour_minutes, reverse=True):
        radius_m = minutes * WALK_SPEED_M_PER_MIN
        subgraph = nx.ego_graph(G, center, radius=radius_m, distance="length")
        nodes_gdf = convert.graph_to_gdfs(subgraph, edges=False)

        # Concave hull via buffered union of reachable nodes
        points = MultiPoint(list(nodes_gdf.geometry))
        walkshed = unary_union(points.buffer(0.0012))  # ~130m buffer for block coverage

        # mapping() returns tuples; normalize to lists for JSON compat
        geom = json.loads(json.dumps(mapping(walkshed)))
        results.append({
            "minutes": minutes,
            "geometry": geom,
        })
    return results


if OUTPUT.exists():
    geojson = json.loads(OUTPUT.read_text())
    # Check if it was generated by OSMnx (has walksheds list) or old Mapbox format
    sample_props = geojson["features"][0]["properties"] if geojson["features"] else {}
    if "walksheds" in sample_props and isinstance(sample_props["walksheds"], list) and len(sample_props["walksheds"]) == len(CONTOUR_MINUTES):
        print(f"Loaded cached {OUTPUT.name} ({len(geojson['features'])} stations)")
    else:
        print(f"Stale cache format detected — regenerating...")
        OUTPUT.unlink()

if not OUTPUT.exists():
    feat_list = []
    for i, station in enumerate(stations):
        print(f"  [{i+1}/{len(stations)}] {station['name']}...", end=" ", flush=True)
        t0 = time.time()
        walksheds = compute_walksheds(station["lat"], station["lon"])
        print(f"ok ({time.time() - t0:.1f}s)")
        feat_list.append({
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [station["lon"], station["lat"]]},
            "properties": {
                "name": station["name"],
                "walksheds": walksheds,
            },
        })

    geojson = {"type": "FeatureCollection", "features": feat_list}
    OUTPUT.parent.mkdir(exist_ok=True)
    OUTPUT.write_text(json.dumps(geojson, indent=2))
    print(f"\nSaved to {OUTPUT}")

print(f"\n{len(geojson['features'])} stations with {len(CONTOUR_MINUTES)} contours each")

  [1/38] Angle Lake... ok (2.5s)
  [2/38] Beacon Hill... ok (9.9s)
  [3/38] BelRed... ok (11.6s)
  [4/38] Bellevue Downtown... ok (30.9s)
  [5/38] Capitol Hill... 

## Interactive Map

Leaflet map with station markers and walkshed polygons (5/10/15-min contours).

In [ ]:
from IPython.display import HTML

CONTOUR_COLORS = {5: "#1a9850", 10: "#fee08b", 15: "#d73027"}
CONTOUR_OPACITY = {5: 0.35, 10: 0.25, 15: 0.15}

# Build walkshed polygon features for Leaflet
walkshed_features = []
for feature in geojson["features"]:
    props = feature["properties"]
    name = props["name"]
    # Handle both old format (single walkshed) and new format (list of walksheds)
    walksheds = props.get("walksheds") or []
    if not walksheds and "walkshed" in props:
        walksheds = [{"minutes": 10, "geometry": props["walkshed"]}]
    for ws in walksheds:
        minutes = ws["minutes"]
        walkshed_features.append({
            "type": "Feature",
            "geometry": ws["geometry"],
            "properties": {
                "name": name,
                "minutes": minutes,
                "color": CONTOUR_COLORS.get(minutes, "#999"),
                "opacity": CONTOUR_OPACITY.get(minutes, 0.2),
            },
        })

# Build station point features
station_features = []
for feature in geojson["features"]:
    station_features.append({
        "type": "Feature",
        "geometry": feature["geometry"],
        "properties": {"name": feature["properties"]["name"]},
    })

walkshed_geojson_str = json.dumps({"type": "FeatureCollection", "features": walkshed_features})
station_geojson_str = json.dumps({"type": "FeatureCollection", "features": station_features})

html = f"""
<div id="map" style="width:100%; height:700px; border-radius:8px;"></div>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script>
(function() {{
    var map = L.map('map').setView([47.58, -122.33], 11);
    L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png', {{
        attribution: '&copy; <a href="https://carto.com/">CARTO</a> &copy; <a href="https://www.openstreetmap.org/copyright">OSM</a>',
        maxZoom: 19,
    }}).addTo(map);

    // Walkshed polygons — render largest first so smaller ones layer on top
    var walksheds = {walkshed_geojson_str};
    var order = {{15: 0, 10: 1, 5: 2}};
    walksheds.features.sort(function(a, b) {{
        return (order[a.properties.minutes] || 0) - (order[b.properties.minutes] || 0);
    }});
    L.geoJSON(walksheds, {{
        style: function(f) {{
            return {{
                color: f.properties.color,
                weight: 1,
                fillColor: f.properties.color,
                fillOpacity: f.properties.opacity,
            }};
        }},
        onEachFeature: function(f, layer) {{
            layer.bindTooltip(f.properties.name + ' (' + f.properties.minutes + ' min walk)');
        }}
    }}).addTo(map);

    // Station markers
    var stations = {station_geojson_str};
    L.geoJSON(stations, {{
        pointToLayer: function(f, latlng) {{
            return L.circleMarker(latlng, {{
                radius: 6, fillColor: '#2563eb', color: '#fff',
                weight: 2, fillOpacity: 0.9,
            }});
        }},
        onEachFeature: function(f, layer) {{
            layer.bindPopup('<b>' + f.properties.name + '</b>');
        }}
    }}).addTo(map);

    // Legend
    var legend = L.control({{position: 'bottomright'}});
    legend.onAdd = function() {{
        var div = L.DomUtil.create('div');
        div.style.cssText = 'background:white;padding:10px 14px;border-radius:6px;font:13px/1.8 sans-serif;box-shadow:0 1px 4px rgba(0,0,0,.3)';
        div.innerHTML = '<b>Walk time</b><br>' +
            '<span style="display:inline-block;width:14px;height:14px;background:#1a9850;border-radius:3px;margin-right:6px;vertical-align:middle"></span>5 min<br>' +
            '<span style="display:inline-block;width:14px;height:14px;background:#fee08b;border-radius:3px;margin-right:6px;vertical-align:middle"></span>10 min<br>' +
            '<span style="display:inline-block;width:14px;height:14px;background:#d73027;border-radius:3px;margin-right:6px;vertical-align:middle"></span>15 min';
        return div;
    }};
    legend.addTo(map);
}})();
</script>
"""
HTML(html)